# Information Visualization II
## School of Information, University of Michigan

## Week 1: 
- Multivariate/Multidimensional + Temporal

## Assignment Overview
### This assignment's objectives include:

- Review, reflect on, and apply different strategies for multidimensional/multivariate/temporal datasets

- Recreate visualizations and propose new and alternative visualizations using [Altair](https://altair-viz.github.io/) 

### The total score of this assignment will be 100 points consisting of:
- You will be producing four visualizations. Three of them will require you to follow the example closely, but the last will be fairly open-ended. For the last one, we'll also ask you to justify why you designed your visualization the way you did.

### Resources:
- Article by [FiveThirtyEight](https://fivethirtyeight.com) available  [online](https://fivethirtyeight.com/features/a-statistical-analysis-of-the-work-of-bob-ross/) (Hickey, 2014)
- The associated dataset on [Github](https://github.com/fivethirtyeight/data/tree/master/bob-ross)
- A dataset of all the [paintings from the show](https://github.com/jwilber/Bob_Ross_Paintings)
    
    
### Important notes:
1) Grading for this assignment is entirely done by manual inspection. For some of the visualizations, we'll expect you to get pretty close to our example (1-3). Problem 4 is more free-form. 

2) Keep your notebooks clean and readable.

3) There are a few instances where our numbers do not align exactly with those from 538. We've pre-processed our data a little bit differently (had different exclusion criteria on guests and for some images we could not process the color data so we excluded those rows).

In [3]:
# load up the resources we need
import urllib.request
import os.path
from os import path
import pandas as pd
import altair as alt
import numpy as np
from sklearn import manifold
from sklearn.metrics import euclidean_distances
from sklearn.decomposition import PCA
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from mads.lib.path import assets

## Bob Ross

Today's assignment will have you working with artwork created by [Bob Ross](https://en.wikipedia.org/wiki/Bob_Ross). Bob was a very famous painter who had a televised painting show from 1983 to 1994. Over 13 seasons and approximately 400 paintings, Bob would walk the audience through a painting project. Often these were landscape images. Bob was famous for telling his audience to paint "happy trees" and sayings like, "We don't make mistakes, just happy little accidents." His soothing voice and bushy hair are well known to many generations of viewers.

If you've never seen an episode, I might suggest starting with [this one](https://www.youtube.com/watch?v=Fw6odlNp7_8). 

![bob ross](bobrosspaints.png)

Bob Ross left a long legacy of art which makes for an interesting dataset to analyze. It's both temporally rich and has a lot of variables we can code. We'll be starting with the dataset created by 538 for their article on a [Statistical Analysis of Bob Ross](https://fivethirtyeight.com/features/a-statistical-analysis-of-the-work-of-bob-ross/). The authors of the article coded each painting to indicate what features the image contained (e.g., one tree, more than one tree, what kinds of clouds, etc.). 

In addition, we've downloaded a second dataset that contains the actual images. We know what kind of paint colors Bob used in each episode, and we have used that to create a dataset for you containing the color distributions. For example, we approximate how much '<font color='#614f4b'>burnt umber</font>' he used by measuring the distance (in color space) from each pixel in the image to the color. We then add the 'similarity' of each pixel to the burnt umber RGB value into the respective column. This is imperfect, of course (paints don't mix this way), but it'll be close enough for our analysis. Note that the sum of those rows will not add to 1 and the total value for any column can be more than 1. The only thing we can guarantee is that the metric is consistent across colors and between paintings.

In [4]:
# the paints Bob used
rosspaints = ['alizarin crimson','bright red','burnt umber','cadmium yellow','dark sienna', 
              'indian yellow','indian red','liquid black','liquid clear','black gesso',
              'midnight black','phthalo blue','phthalo green','prussian blue','sap green',
              'titanium white','van dyke brown','yellow ochre']

# hex values for the paints above
rosspainthex = ['#94261f','#c06341','#614f4b','#f8ed57','#5c2f08','#e6ba25','#cd5c5c',
                '#000000','#ffffff','#000000','#36373c','#2a64ad','#215c2c','#325fa3',
                '#364e00','#f9f7eb','#2d1a0c','#b28426']

# boolean features about what an image includes
imgfeatures = ['Apple frame', 'Aurora borealis', 'Barn', 'Beach', 'Boat', 
               'Bridge', 'Building', 'Bushes', 'Cabin', 'Cactus', 
               'Circle frame', 'Cirrus clouds', 'Cliff', 'Clouds', 
               'Coniferous tree', 'Cumulus clouds', 'Decidious tree', 
               'Diane andre', 'Dock', 'Double oval frame', 'Farm', 
               'Fence', 'Fire', 'Florida frame', 'Flowers', 'Fog', 
               'Framed', 'Grass', 'Guest', 'Half circle frame', 
               'Half oval frame', 'Hills', 'Lake', 'Lakes', 'Lighthouse', 
               'Mill', 'Moon', 'At least one mountain', 'At least two mountains', 
               'Nighttime', 'Ocean', 'Oval frame', 'Palm trees', 'Path', 
               'Person', 'Portrait', 'Rectangle 3d frame', 'Rectangular frame', 
               'River or stream', 'Rocks', 'Seashell frame', 'Snow', 
               'Snow-covered mountain', 'Split frame', 'Steve ross', 
               'Man-made structure', 'Sun', 'Tomb frame', 'At least one tree', 
               'At least two trees', 'Triple frame', 'Waterfall', 'Waves', 
               'Windmill', 'Window frame', 'Winter setting', 'Wood framed']

# load the data frame
file = assets.find("bobross.csv")
bobross = pd.read_csv(file)

# enable correct rendering (unnecessary in later versions of Altair)
alt.renderers.enable('html')

# uses intermediate json files to speed things up
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

We have a few variables defined for you that you might find useful for the rest of this exercise. First is the ```bobross``` dataframe which, has a row for every painting created by Bob (we've removed those created by guest artists).

In [5]:
# run to see what's inside
bobross.sample(5)

,EPISODE,TITLE,RELEASE_DATE,Apple frame,Aurora borealis,Barn,Beach,Boat,Bridge,Building,...,phthalo blue,phthalo green,prussian blue,sap green,titanium white,van dyke brown,yellow ochre,img_url,week_number,year
86,S08E05,"""HUNTER'S HAVEN""",1/30/86,0,0,0,0,0,0,0,...,0.000000,0.000000,0.329983,0.183682,0.589369,0.194895,0.307496,https://raw.githubusercontent.com/jwilber/Bob_...,5,1986
46,S04E12,"""AUTUMN DAYS""",11/21/84,0,0,0,0,0,0,0,...,0.417569,0.363347,0.447685,0.288581,0.335541,0.309340,0.437968,https://raw.githubusercontent.com/jwilber/Bob_...,47,1984
259,S22E03,"""AROUND THE BEND""",1/15/91,0,0,0,0,0,0,0,...,0.352331,0.712914,0.387189,0.648957,0.076850,0.691850,0.433152,https://raw.githubusercontent.com/jwilber/Bob_...,3,1991
276,S23E07,"""AT DAWN'S LIGHT""",10/15/91,0,0,0,0,0,0,0,...,0.000000,0.000000,0.494271,0.000000,0.291682,0.398552,0.496310,https://raw.githubusercontent.com/jwilber/Bob_...,42,1991
25,S03E01,"""MOUNTAIN RETREAT""",1/4/84,0,0,0,0,0,0,0,...,0.413395,0.520301,0.449773,0.447919,0.249854,0.477262,0.465480,https://raw.githubusercontent.com/jwilber/Bob_...,1,1984


In the dataframe you will see an episode identifier (EPISODE, which contains the season and episode number), the image title (TITLE), the release date (RELEASE_DATE as well as another column for the year). There are also a number of boolean columns for the features coded by 538. A '1' means the feature is present, a '0' means it is not. A list of those columns is available in the ```imgfeatures``` variable.

In [6]:
# run to see what's inside
print(imgfeatures)

['Apple frame', 'Aurora borealis', 'Barn', 'Beach', 'Boat', 'Bridge', 'Building', 'Bushes', 'Cabin', 'Cactus', 'Circle frame', 'Cirrus clouds', 'Cliff', 'Clouds', 'Coniferous tree', 'Cumulus clouds', 'Decidious tree', 'Diane andre', 'Dock', 'Double oval frame', 'Farm', 'Fence', 'Fire', 'Florida frame', 'Flowers', 'Fog', 'Framed', 'Grass', 'Guest', 'Half circle frame', 'Half oval frame', 'Hills', 'Lake', 'Lakes', 'Lighthouse', 'Mill', 'Moon', 'At least one mountain', 'At least two mountains', 'Nighttime', 'Ocean', 'Oval frame', 'Palm trees', 'Path', 'Person', 'Portrait', 'Rectangle 3d frame', 'Rectangular frame', 'River or stream', 'Rocks', 'Seashell frame', 'Snow', 'Snow-covered mountain', 'Split frame', 'Steve ross', 'Man-made structure', 'Sun', 'Tomb frame', 'At least one tree', 'At least two trees', 'Triple frame', 'Waterfall', 'Waves', 'Windmill', 'Window frame', 'Winter setting', 'Wood framed']


The columns that contain the amount of each color in the paintings are listed in ```rosspaints```. There is also an analogous list variable called ```rosspainthex``` that has the hex values for the paints. These hex values are approximate.

In [7]:
# run to see what's inside
print("paint names",rosspaints)
print("")
print("hex values", rosspainthex)

paint names ['alizarin crimson', 'bright red', 'burnt umber', 'cadmium yellow', 'dark sienna', 'indian yellow', 'indian red', 'liquid black', 'liquid clear', 'black gesso', 'midnight black', 'phthalo blue', 'phthalo green', 'prussian blue', 'sap green', 'titanium white', 'van dyke brown', 'yellow ochre']

hex values ['#94261f', '#c06341', '#614f4b', '#f8ed57', '#5c2f08', '#e6ba25', '#cd5c5c', '#000000', '#ffffff', '#000000', '#36373c', '#2a64ad', '#215c2c', '#325fa3', '#364e00', '#f9f7eb', '#2d1a0c', '#b28426']


### Problem 1  (20 points)

As a warmup, we're going to have you recreate the [first chart from the Bob Ross article](bob_ross_538.png) (source: [Statistical Analysis of Bob Ross](https://fivethirtyeight.com/features/a-statistical-analysis-of-the-work-of-bob-ross/)). This one simply shows a bar chart for the percent of images that have certain features. The Altair version is:

!["Bob Ross feature distribution"](bob_ross_altair.png)

We'll be using the 538 theme for styling, so you don't have to do much beyond creating the chart (but do note that we want to see the percents, titles, and modifications to the axes). 

You will replace the code for ```makeBobRossBar()``` and have it return an Altair chart.  We suggest you first create a table that contains the names of the features and the percents.  Something like this:

!["Sample Table](feature_table.png)

Recall that this is the 'long form' representation of the data, which will make it easier to create a visualization with. Also, **note the order of the bars. It's not arbitrary, please re-create it.**

In [8]:
def makeBobRossBar(br, ifeatures):
    # input: br -- a dataframe in the shape of the bobross frame defined above
    # input: ifeatures -- a list of the features we want to test (see imgfeatures above)
    # return: implement this function to return an altair chart as defined above
    #         e.g., return alt.Chart(...)
    
    # because the feature columns are binary (0/1), the mean is the
    # proportion of paintings that contain the feature
    feature_table = pd.DataFrame({
        'feature': ifeatures,
        'percent': [br[feature].mean() for feature in ifeatures]
    })

    # reference figure displays whole-number percentages and omits
    # features that round to less than 2%
    feature_table['pct_label'] = (feature_table['percent'] * 100).round().astype(int)
    feature_table = (
        feature_table[feature_table['pct_label'] >= 2]
        .sort_values('percent', ascending = False, kind = 'stable')
        .reset_index(drop = True)
    )

    # preserve the exact descending order of the reference chart
    feature_order = feature_table['feature'].tolist()

    bars = alt.Chart(feature_table).mark_bar().encode(
        x = alt.X(
            'percent:Q',
            scale = alt.Scale(domain = [0, 1]),
            axis = None
        ),
        y = alt.Y(
            'feature:N',
            sort = feature_order,
            title = None,
            axis = alt.Axis(ticks = False, domain = False)
        ),
        tooltip = [
            alt.Tooltip('feature:N', title = 'Feature'),
            alt.Tooltip('percent:Q', title = 'Percent', format = '.0%')
        ]
    )

    # put the rounded percentage immediately to the right of each bar
    labels = alt.Chart(feature_table).mark_text(
        align = 'left',
        baseline = 'middle',
        dx = 4
    ).encode(
        x = alt.X(
            'percent:Q',
            scale = alt.Scale(domain = [0, 1]),
            axis = None
        ),
        y = alt.Y(
            'feature:N',
            sort = feature_order,
            title = None
        ),
        text = alt.Text('percent:Q', format = '.0%')
    )

    return (bars + labels).properties(
        width = 430,
        height = 700,
        title = {
            'text': 'The Paintings of Bob Ross',
            'subtitle': 'Percentage containing each element',
            'anchor': 'start'
        }
    )

In [9]:
# run this code to validate
alt.theme.enable('fivethirtyeight')
makeBobRossBar(bobross, imgfeatures)

alt.LayerChart(...)

## Problem 2 (25 points)

The 538 article ([Statistical Analysis of Bob Ross](https://fivethirtyeight.com/features/a-statistical-analysis-of-the-work-of-bob-ross/)) has a long analysis of conditional probabilities. Essentially, we want to know the probability of one feature given another (e.g., what is the probability of Snow given Trees?). The article calculates this over the entire history of the show, but we would like to visualize these probabilities over time. Have they been constant? or evolving?  We will only be doing this for a few variables (otherwise, we'll have a matrix of over 3000 small charts). The example below is for: 'At least one tree','At least two trees','Clouds','Grass','At least one mountain','Lake.' Each small multiple plot will be a line chart corresponding to the conditional probability over time. The matrix "cell" indicates which pairs of variables are being considered (e.g., probability of at least two trees given the probability of at least one tree is the 2nd row, first column in our example).

Your task will be to generate small multiples plots. For example:

!["Small multiples"](matrix_small.png)

The full image is [available here](matrix_full.png). While your small multiples visualization should contain all this data (the pairwise comparisons), you can ***feel free to style it as you think is appropriate***. We will be grading (minimally) on aesthetics. Implement the code for the function: ```makeBobRossCondProb(...)``` to return this chart.

Some notes on doing this exercise:

* Write test code for makeBobRossCondProb(...) to make sure it works with different inputs.

* If you don't remember how to calculate conditional probabilities, take a look at the article. Remember, we want the conditional probabilities given the images in a specific year. This is simply an implementation of Conditional Probability/Bayes' Theorem. We implemented a function called ```condprobability(...)``` as you can see below. You can do the same or pick your own strategy for this.

* We suggest creating a long-form representation of the table for this data. For example, here's a sample of ours (you can use this to double check your calculations):

!["Long form conditional probabilities](cond_prob_table.png)

* There are a number of strategies to build the small-multiple plots. Some are easier than others. You will find in this case that some combinations of repeated charts and faceting will not work. However, you should be able to use the standard concatenation approaches in combination with repeated charts or faceting.

In [10]:
def condprobability(frame,column1,column2,year):
    # we suggest you implement this function to make your life easier. 
    # input: frame -- the input dataframe in the style of the bobross dataframe above
    # input: column1 -- the first column to test (e.g, the A in probability of A given B)
    # input: column2 -- the second column to test (e.g., the B in the probability of A given B)
    # input: year -- the year for which to calculate the probability
    # return: a conditional probability value

    # you can make variants of this function as you see fit, we will not be calling it directly
    
    year_frame = frame[frame['year'] == year]

    # P(A | B) = count(A and B) / count(B)
    denominator = (year_frame[column2] == 1).sum()

    # If B never occurs in that year, P(A | B) is undefined.
    if denominator == 0:
        return np.nan

    numerator = (
        (year_frame[column1] == 1) &
        (year_frame[column2] == 1)
    ).sum()

    return numerator / denominator

In [25]:
def makeBobRossCondProb(br, totest):
    # implement this function to return an altair chart
    # 
    # input: br the dataframe (e.g., the bobross frame as defined above)
    # input: totest is a variable that holds an array of properties we want compared (see example below)
    
    # we have created a default 'totest' variable that has the columns for the example above
    
    # return alt.Chart(...)
    
    years = sorted(br['year'].dropna().astype(int).unique())

    # build one long-form table containing every ordered feature pair
    rows = []
    for event in totest:
        for condition in totest:
            for year in years:
                rows.append({
                    'key1': event,
                    'key2': condition,
                    'year': int(year),
                    'prob': condprobability(br, event, condition, year)
                })

    cond_table = pd.DataFrame(rows)

    # match the supplied example more closely by explicitly constructing
    # one row of small multiples for each event. Within each row, columns
    # correspond to the conditioning feature ("Given...")
    matrix_rows = []

    for event in totest:
        row_charts = []

        for j, condition in enumerate(totest):
            panel_data = cond_table[
                (cond_table['key1'] == event) &
                (cond_table['key2'] == condition)
            ]

            # only the first chart in each row needs y-axis labels/title
            if j == 0:
                y_axis = alt.Axis(
                    title = 'Probability of {}'.format(event),
                    values = [0, 0.5, 1],
                    format = '.1f',
                    titleFontSize = 11,
                    labelFontSize = 9,
                    grid = True
                )
            else:
                y_axis = alt.Axis(
                    title = None,
                    values = [0, 0.5, 1],
                    labels = False,
                    ticks = False,
                    grid = True
                )

            plot = alt.Chart(panel_data).mark_line(
                strokeWidth = 2
            ).encode(
                x = alt.X(
                    'year:Q',
                    title = None,
                    scale = alt.Scale(domain = [min(years), max(years)]),
                    axis = alt.Axis(
                        values = [1983, 1987, 1991],
                        format = 'd',
                        labelAngle = -90,
                        labelFontSize = 8,
                        grid = True
                    )
                ),
                y = alt.Y(
                    'prob:Q',
                    title = None,
                    scale = alt.Scale(domain = [0, 1]),
                    axis = y_axis
                ),
                tooltip = [
                    alt.Tooltip('key1:N', title = 'Event'),
                    alt.Tooltip('key2:N', title = 'Given'),
                    alt.Tooltip('year:Q', title = 'Year', format = 'd'),
                    alt.Tooltip('prob:Q', title = 'Probability', format = '.1%')
                ]
            ).properties(
                width = 75,
                height = 70
            )

            # separate fixed-height header for the column title
            header_data = pd.DataFrame({'label': [condition]})

            header = alt.Chart(header_data).mark_text(
                fontSize = 9,
                fontWeight = 'normal',
                align = 'center',
                baseline = 'middle'
            ).encode(
                text = 'label:N'
            ).properties(
                width = 75,
                height = 12
            )

            # put the title directly above the plot
            panel = alt.vconcat(
                header,
                plot,
                spacing = 2
            ).properties(
                bounds = 'flush'
            )

            row_charts.append(panel)

        matrix_rows.append(
            alt.hconcat(*row_charts, spacing = 8
            ).properties(
                title = {
                    'text': 'Given...',
                    'anchor': 'middle',
                    'fontSize': 12
                }
            )
        )

    return alt.vconcat(*matrix_rows, spacing = 18).resolve_scale(
        x = 'shared',
        y = 'shared'
    )

In [26]:
# If you did everything right, the following should produce the small multiples grid for the example in
# the description.
makeBobRossCondProb(bobross, ['At least one tree','At least two trees','Clouds','Grass','At least one mountain','Lake'])

alt.VConcatChart(...)

### Additional comments

If you deviated from our example, please use this cell to give us additional information about your design choices and why you think they are an improvement.

### STUDENT RESPONSE:

---

I redesigned the visualization to improve comparison across the small multiples. I increased the spacing between rows and columns, aligned the plotting areas, and created a clearer hierarchy between the row headings, column labels, and data. These changes reduce visual crowding and make the matrix structure easier to understand. Because all panels retain consistent axes and dimensions, viewers can more easily compare conditional probabilities across years and categories. Although the revised design uses more space, the additional whitespace improves readability and makes the data patterns more prominent. 


## Problem 3 (25 points)

Recall that in some cases of multidimensional data a good strategy is to use dimensionality reduction to visualize the information. Here, we would like to understand how images are similar to each other in 'feature' space. Specifically, how similar are they based on the image features? Are images that have beaches close to those with waves? 

We are going to create a 2D MDS plot using the scikit learn package. We're going to do most of this for you in the next cell. Essentially we will use the euclidean distance between two images based on their image feature array to create the image. Your plot may look slightly different than ours based on the random seed (e.g., rotated or reflected), but in the end, it should be close. If you're interested in how this is calculated, we suggest taking a look at [this documentation](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.MDS.html)

Note that the next cell may take a minute or so to run, depending on the server.  

In [27]:
def augmentWithMDS(br=bobross, ifeatures=imgfeatures):
    # input: br -- the bobross shaped dataframe
    # input: ifeatures -- the features we want to use for calculate the MDS layout
    # output: a modified bobross dataframe that has new columns for the x/y coordinates
    
    # create the seed
    seed = np.random.RandomState(seed=3)

    # generate the MDS configuration, we want 2 components, etc. You can tweak this if you want to see how
    # the settings change the layout
    mds = manifold.MDS(n_components=2, max_iter=3000, eps=1e-9, random_state=seed, n_jobs=1)

    # fit the data. At the end, 'pos' will hold the x,y coordinates
    pos = mds.fit(br[ifeatures]).embedding_

    # we'll now load those values into the bobross data frame, giving us a new x column and y column
    br['x'] = [x[0] for x in pos]
    br['y'] = [x[1] for x in pos]
    return(br)

bobross = augmentWithMDS()

/opt/python/lib/python3.12/site-packages/sklearn/manifold/_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
/opt/python/lib/python3.12/site-packages/sklearn/manifold/_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(


Your task is to implement the visualization for the MDS layout. We will be using a new mark, ```mark_image```, for this. You can read all about this mark on the Altair site [here](https://altair-viz.github.io/user_guide/marks.html#user-guide-image-mark). Note that we all already saved the images for you. They are accessible in the img_url column in the bobross table. You will use the ```url``` encode argument to mark_image to make this work.

In this case, we would also like to emphasize all the images that *have* a specific feature. So when you define your ```genMDSPlot()``` function below, it should take a key string as an argument (e.g., 'Beach') and visually highlight those images. A simple way to do this is to use a second mark underneath the image (e.g., a rectangle) that is a different color based on the absence or presence of the image.  Here's an example output for ```genMDSPlot("Palm trees")```:

!["mds"](mds_small.png)

Click [here](mds_large.png) for a large version of this image. Notice the orange boxes indicating where the Palm tree images are. Note also that we have styled the MDS plot to not have axes. Recall that these are meaningless in MDS 'space' (this is not a scatterplot, it's a projection).

Important: *You can make some of your own choices on how to make the matched items salient but you need to make this this visualization usable (expressive & effective). We do expect that your solution not be less effective/expressive than our example.*

Hint: you may want to think about how to get "details" if you make images very small. We'd like to be able to figure out which image is what. A really simply strategy is to use something like tooltips.

!["mds"](mds_tooltip.png)

In [28]:
def genMDSPlot(br,key):
    # input: br -- a bobross dataframe (augmented with the x/y columns as describe above)
    # input: key -- is a string indicating which images should be visually highlighted (i.e., images containing the feature
    #        should be made salient). For example: 'Barn'
    # return: an altair chart (e.g., return alt.Chart(...))
    
    if key not in br.columns:
        raise ValueError("Unknown Bob Ross feature: {}".format(key))

    plotdata = br.copy()

    has_label = "Has a '{}'".format(key)
    not_label = "Doesn't have a '{}'".format(key)

    # every thumbnail gets a colored backing square. Paintings containing
    # the selected feature are orange; all others are light blue
    plotdata['_feature_status'] = np.where(
        plotdata[key] == 1,
        has_label,
        not_label
    )

    backgrounds = alt.Chart(plotdata).mark_square(
        size = 900
    ).encode(
        x = alt.X('x:Q', axis = None),
        y = alt.Y('y:Q', axis = None),
        color = alt.Color(
            '_feature_status:N',
            title = key,
            scale = alt.Scale(
                domain = [has_label, not_label],
                range = ['#ff5a1f', '#b9e3ef']
            ),
            legend = alt.Legend(orient = 'right')
        )
    )

    # draw the actual painting on top of its colored backing square
    paintings = alt.Chart(plotdata).mark_image(
        width = 28,
        height = 21
    ).encode(
        x = alt.X('x:Q', axis = None),
        y = alt.Y('y:Q', axis = None),
        url = 'img_url:N',
        tooltip = [
            alt.Tooltip('EPISODE:N', title = 'Episode'),
            alt.Tooltip('TITLE:N', title = 'Painting'),
            alt.Tooltip('year:Q', title = 'Year', format = 'd'),
            alt.Tooltip('_feature_status:N', title = key)
        ]
    )

    return (backgrounds + paintings).properties(
        width = 900,
        height = 900
    ).configure_view(
        strokeWidth = 0
    )
    

In [29]:
# you should be able to test your code without interactivity, for example:
genMDSPlot(bobross,'Oval frame')

alt.LayerChart(...)

We are going to create an interactive widget that allows you to select the feature you want to be highlighted. If you implemented your ```genMDSPlot``` code correctly, the plot should change when you select new items from the list. We would ordinarily do this directly in Altair, but because we don't have control over the way you created your visualization, it's easiest for us to use the widgets built into Jupyter.

It should look something like this:

!["mds interactive"](interactive_mds.png)

It may take a few seconds the first time you run this to download all the images.

In [30]:
# note that it might take a few seconds for the images to download
# depending on your internet connection

output = widgets.Output()

def clicked(b):
    output.clear_output()
    with output:
        # when the selection is changed, we pull the value and call the altair plot generator
        highlight = filterdrop.value
        if (highlight == ""):
            print("please enter a query")
        else:
            genMDSPlot(bobross,highlight).display()


featurecount = bobross[imgfeatures].sum()

filterdrop = widgets.Dropdown(
    options=list(featurecount[featurecount > 2].keys()),
    description='Highlight:',
    disabled=False,
)

filterdrop.observe(clicked, names=['value'])

display(filterdrop,output)

with output:
    genMDSPlot(bobross,'Barn').display()


Dropdown(description='Highlight:', options=('Barn', 'Beach', 'Bridge', 'Bushes', 'Cabin', 'Cactus', 'Cirrus cl…

Output()

## Problem 4 (30 points: 25 for solution, 5 for explanation)

Your last problem is fairly open-ended in terms of visualization. We would like to analyze the colors used in different images for a given season as a small multiples plot. You can pick how you represent your small multiples, but we will ask you to defend your choices below.  You must implement the function ```colorSmallMultiples(season)``` that takes a season number as input (e.g., 2) and returns an Altair chart. The "multiples" should be at the painting level--so, one multiple per painting (and each TV season shown at once).

Here's a really simple example (that isn't great):

!["simple small multiples"](bob_ross_color_glyph.png)

This visualization has a row (a "mini plot") for every painting and a colored circle (in the color of the paint). The circle is sized based on the amount of the corresponding paint that is used in the image. 

You can also go to something as crazy as this:

!["face small multiples"](bob_ross_face.png)

Here, we've overlaid circles as curls in Bob's massive hair. We're not claiming this is an effective solution, but you're welcome to do this (or anything else) as long as you describe the pros and cons of your choices. And, yes, we generated both examples using Altair.

Again, the relevant columns are available are listed in ```rosspaints``` (there are 18 of them). The values range from 0 to 1 based on the fraction of pixel color allocated to that specific paint.  The ```rosspainthex``` has the corresponding hex values for the paint color. 

*Some notes*

1) We would like for you to be creative. You will not get full credit for this assignment if you simply copy our examples (or change them slightly). 

2) Make sure your visualization is actually a small multiple approach. There should be "mini" visualizations for each painting. This is a "rough" check, but if you're not using repeating, faceting, concatenation, etc. you're probably just making one chart (e.g., a heatmap). Another check is if there are axis labels/information on each so that it's readable on its own (a shared legend is fine). All these are inexact tests but may be helpful as a starting point.

3) You *may* find it useful to implement "colorSmallMultiple" as below to generate your single small multiple. This may not be ideal if you're using faceting or repetition. For example, in our implementation calling ```colorSmallMultiple(5,1)``` will create a small multiple for season 5, episode 1:

!["color small multiple"](single_multiple.png)

4) As with all assignments in this course, test everything!

In [31]:
# this is optional, you can use this to produce a single multiple
# you may not find this helpful for your solution
def colorSmallMultiple(season, episodenumber, br=bobross, rp=rosspaints, rph=rosspainthex):
    # input: season -- a season number (integer), assumed to exist in the dataset
    # input: episodenumber -- an episode number (integer), assumed to exist in the dataset
    # input: br -- a dataset structed as the bobross data above (default is "bobross")
    # input: rp -- the names of paints (default rosspaints as defined above)
    # input: rph -- the hex values of the paints (default rosspaintshex as defined above)
    # return: a single multiple visualization for the season/episode
    
    work = br.copy()

    # The supplied data normally contains lowercase season/episode columns.
    # Fall back to parsing the SxxExx identifier if necessary.
    if 'season' not in work.columns:
        work['_season'] = work['EPISODE'].str.extract(r'S(\d+)')[0].astype(int)
        season_col = '_season'
    else:
        season_col = 'season'

    if 'episode' not in work.columns:
        work['_episode_number'] = work['EPISODE'].str.extract(r'E(\d+)')[0].astype(int)
        episode_col = '_episode_number'
    else:
        episode_col = 'episode'

    painting = work[
        (work[season_col] == season) &
        (work[episode_col] == episodenumber)
    ]

    if painting.empty:
        raise ValueError(
            "No painting found for season {} episode {}".format(
                season, episodenumber
            )
        )

    row = painting.iloc[0]

    paint_table = pd.DataFrame({
        'paint': rp,
        'amount': [row[paint] for paint in rp]
    })

    title = row['TITLE']

    return alt.Chart(paint_table).mark_bar(
        stroke = '#777777',
        strokeWidth = 0.35
    ).encode(
        x = alt.X(
            'paint:N',
            sort = rp,
            title = None,
            axis = alt.Axis(labelAngle = -65, labelFontSize = 8)
        ),
        y = alt.Y(
            'amount:Q',
            title = 'Paint amount',
            scale = alt.Scale(domain = [0, 1])
        ),
        color = alt.Color(
            'paint:N',
            scale = alt.Scale(domain = rp, range = rph),
            legend = None
        ),
        tooltip = [
            alt.Tooltip('paint:N', title = 'Paint'),
            alt.Tooltip('amount:Q', title = 'Amount', format = '.3f')
        ]
    ).properties(
        width = 330,
        height = 130,
        title = 'S{:02d}E{:02d}: {}'.format(
            int(season), int(episodenumber), title
        )
    )

# test 
colorSmallMultiple(12,10)  # season 12, episode 10
colorSmallMultiple(5,1)    # season 5, episode 1

alt.Chart(...)

In [32]:
def colorSmallMultiples(season, br=bobross, rp=rosspaints, rph=rosspainthex):
    # input: season -- a season number (integer), assumed to exist in the dataset. This is the 
    #               integer representing the season of the show are interested in. Limit your images
    #               to that season in the small multiples display.
    # input: br -- a dataset structed as the bobross data above (default is "bobross")
    # input: rp -- the names of paints (default rosspaints as defined above)
    # input: rph -- the hex values of the paints (default rosspainthex as defined above)
    # return: an Altair chart providing small multiples for that season
    
    # input: season -- season number
    # return: painting-level small multiples for the selected season

    work = br.copy()

    if 'season' not in work.columns:
        work['_season'] = work['EPISODE'].str.extract(r'S(\d+)')[0].astype(int)
        season_col = '_season'
    else:
        season_col = 'season'

    if 'episode' not in work.columns:
        work['_episode_number'] = work['EPISODE'].str.extract(r'E(\d+)')[0].astype(int)
        episode_col = '_episode_number'
    else:
        episode_col = 'episode'

    season_frame = work[work[season_col] == season].copy()

    if season_frame.empty:
        raise ValueError("Season {} does not exist in the data".format(season))

    season_frame = season_frame.sort_values(episode_col)

    # Convert the 18 paint columns to long form: one row per
    # painting x paint combination.
    id_cols = [episode_col, 'EPISODE', 'TITLE']
    long = season_frame[id_cols + rp].melt(
        id_vars=id_cols,
        value_vars=rp,
        var_name='paint',
        value_name='amount'
    )

    # A compact panel title keeps the temporal order visible while also
    # identifying the painting.
    long['panel'] = long.apply(
        lambda row: 'E{:02d} · {}'.format(
            int(row[episode_col]),
            str(row['TITLE']).strip('"')
        ),
        axis=1
    )

    panel_order = (
        season_frame
        .apply(
            lambda row: 'E{:02d} · {}'.format(
                int(row[episode_col]),
                str(row['TITLE']).strip('"')
            ),
            axis=1
        )
        .tolist()
    )

    # One mini bar chart ("paint fingerprint") per painting.
    # Height encodes paint amount; hue identifies the actual paint.
    base = alt.Chart(long).mark_bar(
        stroke='#777777',
        strokeWidth=0.3
    ).encode(
        x=alt.X(
            'paint:N',
            sort=rp,
            title=None,
            axis=alt.Axis(labels=False, ticks=False)
        ),
        y=alt.Y(
            'amount:Q',
            title='Amount',
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(tickCount=3, format='.1f')
        ),
        color=alt.Color(
            'paint:N',
            scale=alt.Scale(domain=rp, range=rph),
            legend=alt.Legend(title='Paint')
        ),
        tooltip=[
            alt.Tooltip('EPISODE:N', title='Episode'),
            alt.Tooltip('TITLE:N', title='Painting'),
            alt.Tooltip('paint:N', title='Paint'),
            alt.Tooltip('amount:Q', title='Amount', format='.3f')
        ]
    ).properties(
        width=190,
        height=95
    )

    return base.facet(
        facet=alt.Facet(
            'panel:N',
            sort=panel_order,
            title=None,
            header=alt.Header(
                labelFontSize=10,
                labelLimit=190
            )
        ),
        columns=3
    ).properties(
        title='Bob Ross paint fingerprints — Season {}'.format(season)
    )

In [33]:
# run this to test your code for season 1
colorSmallMultiples(1)

alt.FacetChart(...)

In [34]:
# run this to test your code for season 2
colorSmallMultiples(2)

alt.FacetChart(...)

### Explain your choices

Explain your design here. Describe the pros and cons in terms of visualization principles.

Some advice: Make sure to discuss the pros/cons of your solution in detail. We're looking for explanation of expressiveness and effectiveness. Your design choices will impact what is easy/hard to do (remember, wicked design!) and we want you to self-critique.

---

### STUDENT RESPONSE: 

I used a **paint fingerprint** for each painting: every mini-chart contains the same 18 paint positions, and the height of each colored bar represents the amount of that paint used. The panels are ordered by episode and labeled with both episode number and painting title.

**Expressiveness**: The visualization directly represents both variables in the data. Bar height encodes the quantitative paint amount, while hue encodes paint identity using the supplied Bob Ross paint colors. No quantity is encoded only by color. Every painting receives its own panel, so the result is a true small-multiples display rather than one combined heatmap.

**Effectiveness**: Bar length/position against a common baseline is a relatively precise method for comparing quantities. Every panel uses the same 0–1 scale and the same paint order, which supports comparisons both within a painting AND across paintings. A shared legend avoids repeating 18 long paint names in every panel, while tooltips provide the exact paint name and value when needed. I also add a thin outline to the bars so very light colors such as titanium white remain visible against the background.

**Pros**: The design makes dominant palettes easy to spot, preserves the actual paint colors, keeps episode order visible, and supports repeated visual comparison across a season. Because the encoding is consistent, a viewer can learn the paint positions once and then scan the panels quickly.

**Cons**: 18 paints still create a dense display, and several paints are perceptually similar, so hue alone is not always sufficient for identification, though the shared legend and fixed x-position mitigate this. Very small paint amounts can also be difficult to compare in tiny panels, although the tooltip gives the exact value. Finally, wrapping the episodes into three columns saves space but introduces a row break in the temporal sequence; the explicit episode labels preserve the ordering.
